# 🏥 Medical QLoRA + GGUF → Dr. Deerfly UI
> **Complete pipeline:** Fine-tune → Convert to GGUF → Download → Load in Ollama → Connect to Dr. Deerfly!

### ⚠️ Before starting:
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. Click **Allow** when Google Drive asks
4. Do NOT close this tab!

## Step 1 — Keep Colab Alive

In [ ]:
from IPython.display import Javascript, display
display(Javascript("""
function keepAlive() {
  console.log('Keeping alive...');
  document.querySelector('#top-toolbar')?.click();
}
setInterval(keepAlive, 60000);
console.log('Keep-alive started!');
"""))
print('Keep-alive running!')

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/drive/MyDrive/MedicalAI', exist_ok=True)
print('Google Drive mounted!')
print('Save folder: /content/drive/MyDrive/MedicalAI')

## Step 3 — Install Libraries (3-5 mins)

In [ ]:
import subprocess, sys
print('Installing Unsloth and dependencies...')
subprocess.run(['pip', 'install', 'unsloth[colab-new]', '-q'], check=False)
subprocess.run(['pip', 'install', '--upgrade', 'transformers', 'datasets', 'peft', 'trl', '-q'], check=False)
print('All libraries installed!')

## Step 4 — Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    mem = gpu.total_memory / 1e9
    print(f'GPU: {gpu.name}')
    print(f'Memory: {mem:.1f} GB')
    print('Ready for training!')
else:
    print('No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

## Step 5 — Load Model (2-3 mins)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'unsloth/tinyllama-bnb-4bit'
MAX_SEQ = 512

print(f'Loading {MODEL_NAME}...')
print('This is TinyLlama - small and fast, perfect for GGUF conversion!')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)
print('Model loaded!')

## Step 6 — Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj','v_proj','k_proj','o_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing=True,
    random_state=42,
)
print('LoRA adapters added!')
print('Rank: 8, Alpha: 16')

## Step 7 — Load Medical Dataset

In [ ]:
from datasets import load_dataset

print('Loading ChatDoctor medical dataset...')
ds = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k', split='train')

# Use only 300 samples for speed
ds = ds.select(range(300))
print(f'Loaded {len(ds)} medical Q&A samples')

def format_sample(sample):
    q = sample.get('input', sample.get('instruction', ''))
    a = sample.get('output', sample.get('response', ''))
    return {'text': f'Patient: {q}\nDoctor: {a}'}

ds = ds.map(format_sample)
print('Dataset ready!')
print('Sample:')
print(ds[0]['text'][:200])

## Step 8 — Train! (5-8 mins)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print('Starting medical fine-tuning...')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field='text',
    max_seq_length=512,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        output_dir='/content/checkpoints',
        save_strategy='no',
        report_to='none',
    ),
)

trainer.train()
print('Training complete!')

## Step 9 — Save LoRA Adapter to Drive

In [ ]:
ADAPTER_PATH = '/content/drive/MyDrive/MedicalAI/adapter'
import os
os.makedirs(ADAPTER_PATH, exist_ok=True)
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved to Google Drive!')
print(f'Location: {ADAPTER_PATH}')

## Step 10 — Test Your Fine-Tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question):
    prompt = f'Patient: {question}\nDoctor:'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split('Doctor:')[-1].strip()

print('Testing your fine-tuned model...')
print()
q = 'What are the symptoms of diabetes?'
print(f'Question: {q}')
print(f'Answer: {ask(q)}')

## Step 11 — Ask Your Own Question

In [ ]:
# Change this question to anything you want!
MY_QUESTION = 'What causes a heart attack?'

print(f'Question: {MY_QUESTION}')
print(f'Answer: {ask(MY_QUESTION)}')

## Step 12 — Convert to GGUF & Connect to Dr. Deerfly UI!
> This is the most important step! Converts your model to GGUF format for Ollama.
> **Takes 10-15 minutes. Do not close the tab!**

In [ ]:
import os
from google.colab import files

GGUF_PATH = '/content/drive/MyDrive/MedicalAI/gguf'
os.makedirs(GGUF_PATH, exist_ok=True)

print('STEP 12A: Saving model in GGUF format...')
print('This takes 10-15 minutes. Please wait...')

try:
    # Save directly as GGUF without merging first - faster and lighter!
    model.save_pretrained_gguf(
        '/content/gguf_output',
        tokenizer,
        quantization_method='q4_k_m'
    )
    print('GGUF conversion done!')

    # Check file size
    gguf_files = []
    for f in os.listdir('/content/gguf_output'):
        if f.endswith('.gguf'):
            fp = f'/content/gguf_output/{f}'
            size_gb = os.path.getsize(fp) / 1e9
            print(f'File: {f} — {size_gb:.2f} GB')
            gguf_files.append(fp)

    # Copy to Google Drive
    print('Copying to Google Drive...')
    for fp in gguf_files:
        fname = os.path.basename(fp)
        dest = f'{GGUF_PATH}/{fname}'
        os.system(f'cp "{fp}" "{dest}"')
        size = os.path.getsize(dest) / 1e9
        print(f'Saved to Drive: {dest} ({size:.2f} GB)')

    # Download to PC
    print('Downloading to your PC now...')
    for fp in gguf_files:
        files.download(fp)
        print(f'Download started: {os.path.basename(fp)}')

    print()
    print('SUCCESS! GGUF file downloaded!')
    print()
    print('NEXT STEPS ON YOUR PC:')
    print('1. Wait for download to complete in your Downloads folder')
    print('2. Create a Modelfile (no extension) in same folder with:')
    print()
    print('   FROM ./unsloth.Q4_K_M.gguf')
    print('   SYSTEM "You are MedAssist, a medical AI. Answer clearly."')
    print('   PARAMETER temperature 0.7')
    print('   PARAMETER num_ctx 2048')
    print()
    print('3. Open CMD in that folder and run:')
    print('   ollama create medassist -f Modelfile')
    print('   ollama run medassist')
    print('4. Open Dr. Deerfly UI, select medassist from dropdown!')

except Exception as e:
    print(f'Error: {e}')
    print('Trying alternative method...')
    try:
        # Alternative: merge first then convert
        model.save_pretrained_merged('/content/merged', tokenizer, save_method='merged_4bit')
        model.save_pretrained_gguf('/content/gguf_output', tokenizer, quantization_method='q2_k')
        print('Alternative conversion done!')
        for f in os.listdir('/content/gguf_output'):
            if f.endswith('.gguf'):
                files.download(f'/content/gguf_output/{f}')
                print(f'Downloading: {f}')
    except Exception as e2:
        print(f'Both methods failed: {e2}')
        print('Please tell Claude the exact error message!')
